# Importing the packages and data

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from sklearn.metrics import r2_score, mean_squared_error

from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from texttable import Texttable
import latextable

In [3]:
import sys
sys.path.insert(1, '../sar_dirichlet')
import dirichlet_regression

In [4]:
from func_test import cos_similarity, create_features_matrices, rmse_aitchison, cosine_similarity_aitchison, r2_aitchison_adjusted

# Loading Dataset

In [5]:
arctic = pd.read_csv('Data Dirichlet/ArcticLake.csv')

In [6]:
Y_arctic = arctic.iloc[:,:3]
X_arctic = arctic.iloc[:,3]

In [7]:
Y_arctic = np.array(Y_arctic)

In [8]:
X_arctic = np.array([[j] for j in X_arctic])

In [9]:
Z_arctic = np.ones(len(X_arctic)).reshape((39,1))

In [10]:
n_features = 1
n_classes = 3

In [11]:
X = np.array(X_arctic)
Y = np.array(Y_arctic)

X = StandardScaler().fit(X).transform(X)

In [12]:
Z = Z_arctic
gamma_0 = [0.]

In [13]:
n,K = X.shape
J = Y.shape[1]

# Order 1

In [14]:
distance_matrix = scipy.spatial.distance_matrix(X_arctic,X_arctic)
W_arctic_cont = np.zeros(np.shape(distance_matrix))
#W_arctic_cont[distance_matrix < 28] = 1
W_arctic_cont[distance_matrix < 20] = 1
# replace the 1 on the diagonal by 0
np.fill_diagonal(W_arctic_cont,0)
# scaling the matrix, so that the sum of each row is 1
W_arctic_cont = W_arctic_cont/W_arctic_cont.sum(axis=1)[:,None]

In [15]:
neighbors = NearestNeighbors(n_neighbors=15).fit(X)
W_arctic_dist = neighbors.kneighbors_graph(X,mode='distance').toarray()
W_arctic_dist[W_arctic_dist>0] = 1/W_arctic_dist[W_arctic_dist>0]
W_arctic_dist = W_arctic_dist/W_arctic_dist.sum(axis=1)[:,None]

## Dirichlet model

In [19]:
%%time
list_r2_ns_1, list_rmse_ns_1, list_aic_ns_1, list_crossentropy_ns_1, list_similarity_ns_1 = [], [], [], [], []
list_r2_cont_1, list_rmse_cont_1, list_aic_cont_1, list_crossentropy_cont_1, list_similarity_cont_1 = [], [], [], [], []
list_r2_dist_1, list_rmse_dist_1, list_aic_dist_1, list_crossentropy_dist_1, list_similarity_dist_1 = [], [], [], [], []
list_rmse_a_ns_1, list_rmse_a_cont_1, list_rmse_a_dist_1 = [], [], []

for i in range(39):  
    X_temp = np.delete(X,i,axis=0)
    Y_temp = np.delete(Y_arctic,i,axis=0)
    Z_temp = np.delete(Z_arctic,i,axis=0)
    
    Wss_cont = np.delete(W_arctic_cont,i,axis=0)
    Wss_cont = np.delete(Wss_cont,i,axis=1)
    Wss_cont = Wss_cont/Wss_cont.sum(axis=1)[:,None]
    
    Wss_dist = np.delete(W_arctic_dist,i,axis=0)
    Wss_dist = np.delete(Wss_dist,i,axis=1)
    Wss_dist = Wss_dist/Wss_dist.sum(axis=1)[:,None]
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp)
    #list_r2_ns_1.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_ns_1.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,5))
    list_rmse_ns_1.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_ns_1.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    list_aic_ns_1.append(-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_temp)+2*5)
    #list_similarity_ns_1.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_ns_1.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_ns_1.append(rmse_aitchison(Y_temp,dirichRegressor.mu))

    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_cont)
    #list_r2_cont_1.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_cont_1.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,6))
    list_rmse_cont_1.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_cont_1.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    list_aic_cont_1.append(-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_temp)+2*6)
    #list_similarity_cont_1.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_cont_1.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_cont_1.append(rmse_aitchison(Y_temp,dirichRegressor.mu))
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_dist)
    #list_r2_dist_1.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_dist_1.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,6))
    list_rmse_dist_1.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_dist_1.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    list_aic_dist_1.append(-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_temp)+2*6)
    #list_similarity_dist_1.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_dist_1.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_dist_1.append(rmse_aitchison(Y_temp,dirichRegressor.mu))

Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: NO

C:\Users\tnguyen001\AppData\Roaming\Python\Python38\site-packages\scipy\optimize\_numdiff.py:576: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0


CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimiz

C:\Users\tnguyen001\AppData\Roaming\Python\Python38\site-packages\scipy\optimize\_numdiff.py:576: RuntimeWarning: invalid value encountered in subtract
  df = fun(x) - f0


CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Wall time: 6.03 s


In [20]:
table = Texttable()
table.add_row(["Model", "$R^2$", "RMSE", "Cross-entropy", "Cos similarity", "AIC", "RMSE_A"])
text_r2 = str(np.round(np.mean(list_r2_ns_1),4))+" ("+str(np.round(np.std(list_r2_ns_1),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_ns_1),4))+" ("+str(np.round(np.std(list_rmse_ns_1),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_ns_1),4))+" ("+str(np.round(np.std(list_crossentropy_ns_1),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_ns_1),4))+" ("+str(np.round(np.std(list_similarity_ns_1),4))+")"
text_aic = str(np.round(np.mean(list_aic_ns_1),4))+" ("+str(np.round(np.std(list_aic_ns_1),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_ns_1),4))+" ("+str(np.round(np.std(list_rmse_a_ns_1),4))+")"
table.add_row(["Not spatial", text_r2, text_rmse, text_crossentropy, text_similarity, text_aic, text_rmse_a])

text_r2 = str(np.round(np.mean(list_r2_cont_1),4))+" ("+str(np.round(np.std(list_r2_cont_1),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_cont_1),4))+" ("+str(np.round(np.std(list_rmse_cont_1),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_cont_1),4))+" ("+str(np.round(np.std(list_crossentropy_cont_1),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_cont_1),4))+" ("+str(np.round(np.std(list_similarity_cont_1),4))+")"
text_aic = str(np.round(np.mean(list_aic_cont_1),4))+" ("+str(np.round(np.std(list_aic_cont_1),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_cont_1),4))+" ("+str(np.round(np.std(list_rmse_a_cont_1),4))+")"
table.add_row(["Contiguity", text_r2, text_rmse, text_crossentropy, text_similarity, text_aic, text_rmse_a])

text_r2 = str(np.round(np.mean(list_r2_dist_1),4))+" ("+str(np.round(np.std(list_r2_dist_1),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_dist_1),4))+" ("+str(np.round(np.std(list_rmse_dist_1),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_dist_1),4))+" ("+str(np.round(np.std(list_crossentropy_dist_1),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_dist_1),4))+" ("+str(np.round(np.std(list_similarity_dist_1),4))+")"
text_aic = str(np.round(np.mean(list_aic_dist_1),4))+" ("+str(np.round(np.std(list_aic_dist_1),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_dist_1),4))+" ("+str(np.round(np.std(list_rmse_a_dist_1),4))+")"
table.add_row(["Distance", text_r2, text_rmse, text_crossentropy, text_similarity, text_aic, text_rmse_a])

print(table.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Model     | $R^2$     | RMSE     | Cross-   | Cos simi | AIC      | RMSE_A   |
|           |           |          | entropy  | larity   |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
| Not       | 0.5627    | 0.1015   | 0.9134   | 0.8686   | -141.635 | 0.7696   |
| spatial   | (0.0126)  | (0.0018) | (0.0027) | (0.0053) | 4        | (0.0157) |
|           |           |          |          |          | (2.2535) |          |
+-----------+-----------+----------+----------+----------+----------+----------+
| Contiguit | 0.5684    | 0.098    | 0.9092   | 0.8885   | -144.482 | 0.7515   |
| y         | (0.0123)  | (0.0018) | (0.0026) | (0.0046) | 6        | (0.0149) |
|           |           |          |          |          | (2.3295) |          |
+-----------+-----------+----------+----------+----------+----------+----------+
| Distance  | 0.5526    | 0.

In [17]:
# For the computation of the AIC, we take the full dataset

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
dirichRegressor.fit(X, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic)
aic_ns = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*5
print('AIC not spatial:', aic_ns)

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor.fit(X, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic, W=W_arctic_cont)
aic_cont = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*6
print('AIC contiguity:', aic_cont)

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor.fit(X, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic, W=W_arctic_dist)
aic_dist = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*6
print('AIC distance:', aic_dist)

Optimization terminated successfully.
AIC not spatial: -145.45619880071732
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
AIC contiguity: -148.46094834935343
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
AIC distance: -143.57411309123344


## Cross-entropy

In [21]:
%%time
list_r2_ns_1_ce, list_rmse_ns_1_ce, list_crossentropy_ns_1_ce, list_similarity_ns_1_ce = [], [], [], []
list_r2_cont_1_ce, list_rmse_cont_1_ce, list_crossentropy_cont_1_ce, list_similarity_cont_1_ce = [], [], [], []
list_r2_dist_1_ce, list_rmse_dist_1_ce, list_crossentropy_dist_1_ce, list_similarity_dist_1_ce = [], [], [], []
list_rmse_a_ns_1_ce, list_rmse_a_cont_1_ce, list_rmse_a_dist_1_ce = [], [], []

for i in range(39):  
    X_temp = np.delete(X_arctic,i,axis=0)
    Y_temp = np.delete(Y_arctic,i,axis=0)
    Z_temp = np.delete(Z_arctic,i,axis=0)
    
    Wss_cont = np.delete(W_arctic_cont,i,axis=0)
    Wss_cont = np.delete(Wss_cont,i,axis=1)
    Wss_cont = Wss_cont/Wss_cont.sum(axis=1)[:,None]
    
    Wss_dist = np.delete(W_arctic_dist,i,axis=0)
    Wss_dist = np.delete(Wss_dist,i,axis=1)
    Wss_dist = Wss_dist/Wss_dist.sum(axis=1)[:,None]
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, loss='crossentropy')
    #list_r2_ns_1_ce.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_ns_1_ce.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,4))
    list_rmse_ns_1_ce.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_ns_1_ce.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    #list_similarity_ns_1_ce.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_ns_1_ce.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_ns_1_ce.append(rmse_aitchison(Y_temp,dirichRegressor.mu))

    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_cont, loss='crossentropy')
    #list_r2_cont_1_ce.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_cont_1_ce.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,5))
    list_rmse_cont_1_ce.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_cont_1_ce.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    #list_similarity_cont_1_ce.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_cont_1_ce.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_cont_1_ce.append(rmse_aitchison(Y_temp,dirichRegressor.mu))
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_dist, loss='crossentropy')
    #list_r2_dist_1_ce.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_dist_1_ce.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,5))
    list_rmse_dist_1_ce.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_dist_1_ce.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    #list_similarity_dist_1_ce.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_dist_1_ce.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_dist_1_ce.append(rmse_aitchison(Y_temp,dirichRegressor.mu))


Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization

In [23]:
table = Texttable()
table.add_row(["Model", "$R^2$", "RMSE", "Cross-entropy", "Cos similarity", "RMSE_A"])
text_r2 = str(np.round(np.mean(list_r2_ns_1_ce),4))+" ("+str(np.round(np.std(list_r2_ns_1_ce),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_ns_1_ce),4))+" ("+str(np.round(np.std(list_rmse_ns_1_ce),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_ns_1_ce),4))+" ("+str(np.round(np.std(list_crossentropy_ns_1_ce),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_ns_1_ce),4))+" ("+str(np.round(np.std(list_similarity_ns_1_ce),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_ns_1_ce),4))+" ("+str(np.round(np.std(list_rmse_a_ns_1_ce),4))+")"
table.add_row(["Not spatial", text_r2, text_rmse, text_crossentropy, text_similarity, text_rmse_a])

text_r2 = str(np.round(np.mean(list_r2_cont_1_ce),4))+" ("+str(np.round(np.std(list_r2_cont_1_ce),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_cont_1_ce),4))+" ("+str(np.round(np.std(list_rmse_cont_1_ce),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_cont_1_ce),4))+" ("+str(np.round(np.std(list_crossentropy_cont_1_ce),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_cont_1_ce),4))+" ("+str(np.round(np.std(list_similarity_cont_1_ce),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_cont_1_ce),4))+" ("+str(np.round(np.std(list_rmse_a_cont_1_ce),4))+")"
table.add_row(["Contiguity", text_r2, text_rmse, text_crossentropy, text_similarity, text_rmse_a])

text_r2 = str(np.round(np.mean(list_r2_dist_1_ce),4))+" ("+str(np.round(np.std(list_r2_dist_1_ce),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_dist_1_ce),4))+" ("+str(np.round(np.std(list_rmse_dist_1_ce),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_dist_1_ce),4))+" ("+str(np.round(np.std(list_crossentropy_dist_1_ce),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_dist_1_ce),4))+" ("+str(np.round(np.std(list_similarity_dist_1_ce),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_dist_1_ce),4))+" ("+str(np.round(np.std(list_rmse_a_dist_1_ce),4))+")"
table.add_row(["Distance", text_r2, text_rmse, text_crossentropy, text_similarity, text_rmse_a])

print(table.draw())

+-------------+------------+------------+------------+------------+------------+
| Model       | $R^2$      | RMSE       | Cross-     | Cos        | RMSE_A     |
|             |            |            | entropy    | similarity |            |
+-------------+------------+------------+------------+------------+------------+
| Not spatial | 0.5448     | 0.0961     | 0.9106     | 0.8798     | 0.8168     |
|             | (0.0128)   | (0.0016)   | (0.0027)   | (0.0055)   | (0.0178)   |
+-------------+------------+------------+------------+------------+------------+
| Contiguity  | 0.5566     | 0.0937     | 0.9064     | 0.9051     | 0.7762     |
|             | (0.0128)   | (0.0017)   | (0.0025)   | (0.0048)   | (0.0165)   |
+-------------+------------+------------+------------+------------+------------+
| Distance    | 0.5144     | 0.0948     | 0.909      | 0.8867     | 0.8229     |
|             | (0.0188)   | (0.0016)   | (0.0025)   | (0.0054)   | (0.0209)   |
+-------------+------------+

# Order 2

In [24]:
X_2 = np.ones((39,3))
X_2[:,1] = X[:,0]
X_2[:,2] = X[:,0]**2

## Dirichlet model

In [25]:
%%time
list_r2_ns_2, list_rmse_ns_2, list_aic_ns_2, list_crossentropy_ns_2, list_similarity_ns_2 = [], [], [], [], []
list_r2_cont_2, list_rmse_cont_2, list_aic_cont_2, list_crossentropy_cont_2, list_similarity_cont_2 = [], [], [], [], []
list_r2_dist_2, list_rmse_dist_2, list_aic_dist_2, list_crossentropy_dist_2, list_similarity_dist_2 = [], [], [], [], []
list_rmse_a_ns_2, list_rmse_a_cont_2, list_rmse_a_dist_2 = [], [], []

for i in range(39):  
    X_temp = np.delete(X_2,i,axis=0)
    Y_temp = np.delete(Y_arctic,i,axis=0)
    Z_temp = np.delete(Z_arctic,i,axis=0)
    
    Wss_cont = np.delete(W_arctic_cont,i,axis=0)
    Wss_cont = np.delete(Wss_cont,i,axis=1)
    Wss_cont = Wss_cont/Wss_cont.sum(axis=1)[:,None]
    
    Wss_dist = np.delete(W_arctic_dist,i,axis=0)
    Wss_dist = np.delete(Wss_dist,i,axis=1)
    Wss_dist = Wss_dist/Wss_dist.sum(axis=1)[:,None]
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp)
    #list_r2_ns_2.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_ns_2.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,9))
    list_rmse_ns_2.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_ns_2.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    list_aic_ns_2.append(-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_temp)+2*5)
    #list_similarity_ns_2.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_ns_2.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_ns_2.append(rmse_aitchison(Y_temp,dirichRegressor.mu))

    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_cont)
    #list_r2_cont_2.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_cont_2.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,10))
    list_rmse_cont_2.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_cont_2.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    list_aic_cont_2.append(-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_temp)+2*6)
    #list_similarity_cont_2.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_cont_2.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_cont_2.append(rmse_aitchison(Y_temp,dirichRegressor.mu))
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_dist)
    #list_r2_dist_2.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_dist_2.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,10))
    list_rmse_dist_2.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_dist_2.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    list_aic_dist_2.append(-2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_temp)+2*6)
    #list_similarity_dist_2.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_dist_2.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_dist_2.append(rmse_aitchison(Y_temp,dirichRegressor.mu))

Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimization terminated successfully.
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
Optimization terminated successfully.
CONVERGENCE: REL_R

In [28]:
dirichRegressor.beta

array([[ 0.55254179,  0.4309239 ],
       [ 0.55254179,  0.4309239 ],
       [ 0.75074296,  1.2441208 ],
       [-0.40159332, -0.65227182]])

In [26]:
table = Texttable()
table.add_row(["Model", "$R^2$", "RMSE", "Cross-entropy", "Cos similarity", "AIC", "RMSE_A"])
text_r2 = str(np.round(np.mean(list_r2_ns_2),4))+" ("+str(np.round(np.std(list_r2_ns_2),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_ns_2),4))+" ("+str(np.round(np.std(list_rmse_ns_2),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_ns_2),4))+" ("+str(np.round(np.std(list_crossentropy_ns_2),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_ns_2),4))+" ("+str(np.round(np.std(list_similarity_ns_2),4))+")"
text_aic = str(np.round(np.mean(list_aic_ns_2),4))+" ("+str(np.round(np.std(list_aic_ns_2),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_ns_2),4))+" ("+str(np.round(np.std(list_rmse_a_ns_2),4))+")"
table.add_row(["Not spatial", text_r2, text_rmse, text_crossentropy, text_similarity, text_aic, text_rmse_a])

text_r2 = str(np.round(np.mean(list_r2_cont_2),4))+" ("+str(np.round(np.std(list_r2_cont_2),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_cont_2),4))+" ("+str(np.round(np.std(list_rmse_cont_2),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_cont_2),4))+" ("+str(np.round(np.std(list_crossentropy_cont_2),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_cont_2),4))+" ("+str(np.round(np.std(list_similarity_cont_2),4))+")"
text_aic = str(np.round(np.mean(list_aic_cont_2),4))+" ("+str(np.round(np.std(list_aic_cont_2),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_cont_2),4))+" ("+str(np.round(np.std(list_rmse_a_cont_2),4))+")"
table.add_row(["Contiguity", text_r2, text_rmse, text_crossentropy, text_similarity, text_aic, text_rmse_a])

text_r2 = str(np.round(np.mean(list_r2_dist_2),4))+" ("+str(np.round(np.std(list_r2_dist_2),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_dist_2),4))+" ("+str(np.round(np.std(list_rmse_dist_2),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_dist_2),4))+" ("+str(np.round(np.std(list_crossentropy_dist_2),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_dist_2),4))+" ("+str(np.round(np.std(list_similarity_dist_2),4))+")"
text_aic = str(np.round(np.mean(list_aic_dist_2),4))+" ("+str(np.round(np.std(list_aic_dist_2),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_dist_2),4))+" ("+str(np.round(np.std(list_rmse_a_dist_2),4))+")"
table.add_row(["Distance", text_r2, text_rmse, text_crossentropy, text_similarity, text_aic, text_rmse_a])

print(table.draw())

+-----------+-----------+----------+----------+----------+----------+----------+
| Model     | $R^2$     | RMSE     | Cross-   | Cos simi | AIC      | RMSE_A   |
|           |           |          | entropy  | larity   |          |          |
+-----------+-----------+----------+----------+----------+----------+----------+
| Not       | 0.6211    | 0.0881   | 0.8993   | 0.9017   | -168.122 | 0.6278   |
| spatial   | (0.0155)  | (0.0019) | (0.0032) | (0.0057) | 1        | (0.0152) |
|           |           |          |          |          | (3.3212) |          |
+-----------+-----------+----------+----------+----------+----------+----------+
| Contiguit | 0.6148    | 0.0877   | 0.8982   | 0.9098   | -167.451 | 0.6213   |
| y         | (0.0173)  | (0.002)  | (0.003)  | (0.005)  | 6 (3.83) | (0.0159) |
+-----------+-----------+----------+----------+----------+----------+----------+
| Distance  | 0.6092    | 0.0883   | 0.8996   | 0.9011   | -166.521 | 0.6281   |
|           | (0.0176)  | (0

In [31]:
# For the computation of the AIC, we take the full dataset

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
dirichRegressor.fit(X_2, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic)
aic_ns = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*5
print('AIC not spatial:', aic_ns)

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor.fit(X_2, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic, W=W_arctic_cont)
aic_cont = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*6
print('AIC contiguity:', aic_cont)

dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
dirichRegressor.fit(X_2, Y_arctic, parametrization='alternative', gamma_0=gamma_0, Z=Z_arctic, W=W_arctic_dist)
aic_dist = -2*dirichlet_regression.dirichlet_loglikelihood(dirichRegressor.mu,dirichRegressor.phi,Y_arctic)+2*6
print('AIC distance:', aic_dist)

Optimization terminated successfully.
AIC not spatial: -172.56078606783757
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
AIC contiguity: -171.8199385214284
CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH
AIC distance: -170.67331096664614


## Cross-entropy

In [27]:
%%time
list_r2_ns_2_ce, list_rmse_ns_2_ce, list_crossentropy_ns_2_ce, list_similarity_ns_2_ce = [], [], [], []
list_r2_cont_2_ce, list_rmse_cont_2_ce, list_crossentropy_cont_2_ce, list_similarity_cont_2_ce = [], [], [], []
list_r2_dist_2_ce, list_rmse_dist_2_ce, list_crossentropy_dist_2_ce, list_similarity_dist_2_ce = [], [], [], []
list_rmse_a_ns_2_ce, list_rmse_a_cont_2_ce, list_rmse_a_dist_2_ce = [], [], []

for i in range(39):  
    X_temp = np.delete(X_2,i,axis=0)
    Y_temp = np.delete(Y_arctic,i,axis=0)
    Z_temp = np.delete(Z_arctic,i,axis=0)
    
    Wss_cont = np.delete(W_arctic_cont,i,axis=0)
    Wss_cont = np.delete(Wss_cont,i,axis=1)
    Wss_cont = Wss_cont/Wss_cont.sum(axis=1)[:,None]
    
    Wss_dist = np.delete(W_arctic_dist,i,axis=0)
    Wss_dist = np.delete(Wss_dist,i,axis=1)
    Wss_dist = Wss_dist/Wss_dist.sum(axis=1)[:,None]
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=False)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, loss='crossentropy')
    #list_r2_ns_2_ce.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_ns_2_ce.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,8))
    list_rmse_ns_2_ce.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_ns_2_ce.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    #list_similarity_ns_2_ce.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_ns_2_ce.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_ns_2_ce.append(rmse_aitchison(Y_temp,dirichRegressor.mu))

    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_cont, loss='crossentropy')
    #list_r2_cont_2_ce.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_cont_2_ce.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,9))
    list_rmse_cont_2_ce.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_cont_2_ce.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    #list_similarity_cont_2_ce.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_cont_2_ce.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_cont_2_ce.append(rmse_aitchison(Y_temp,dirichRegressor.mu))
    
    dirichRegressor = dirichlet_regression.dirichletRegressor(spatial=True)
    dirichRegressor.fit(X_temp, Y_temp, parametrization='alternative', gamma_0=gamma_0, Z=Z_temp, W=Wss_dist, loss='crossentropy')
    #list_r2_dist_2_ce.append(r2_score(Y_temp,dirichRegressor.mu))
    list_r2_dist_2_ce.append(r2_aitchison_adjusted(Y_temp,dirichRegressor.mu,9))
    list_rmse_dist_2_ce.append(mean_squared_error(Y_temp,dirichRegressor.mu,squared=False))
    list_crossentropy_dist_2_ce.append(-1/n*np.sum(Y_temp*np.log(dirichRegressor.mu)))
    #list_similarity_dist_2_ce.append(cos_similarity(Y_temp,dirichRegressor.mu))
    list_similarity_dist_2_ce.append(cosine_similarity_aitchison(Y_temp,dirichRegressor.mu))
    list_rmse_a_dist_2_ce.append(rmse_aitchison(Y_temp,dirichRegressor.mu))

Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization terminated successfully.
Optimization

In [28]:
table = Texttable()
table.add_row(["Model", "$R^2$", "RMSE", "Cross-entropy", "Cos similarity", "RMSE_A"])
text_r2 = str(np.round(np.mean(list_r2_ns_2_ce),4))+" ("+str(np.round(np.std(list_r2_ns_2_ce),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_ns_2_ce),4))+" ("+str(np.round(np.std(list_rmse_ns_2_ce),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_ns_2_ce),4))+" ("+str(np.round(np.std(list_crossentropy_ns_2_ce),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_ns_2_ce),4))+" ("+str(np.round(np.std(list_similarity_ns_2_ce),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_ns_2_ce),4))+" ("+str(np.round(np.std(list_rmse_a_ns_2_ce),4))+")"
table.add_row(["Not spatial", text_r2, text_rmse, text_crossentropy, text_similarity, text_rmse_a])

text_r2 = str(np.round(np.mean(list_r2_cont_2_ce),4))+" ("+str(np.round(np.std(list_r2_cont_2_ce),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_cont_2_ce),4))+" ("+str(np.round(np.std(list_rmse_cont_2_ce),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_cont_2_ce),4))+" ("+str(np.round(np.std(list_crossentropy_cont_2_ce),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_cont_2_ce),4))+" ("+str(np.round(np.std(list_similarity_cont_2_ce),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_cont_2_ce),4))+" ("+str(np.round(np.std(list_rmse_a_cont_2_ce),4))+")"
table.add_row(["Contiguity", text_r2, text_rmse, text_crossentropy, text_similarity, text_rmse_a])

text_r2 = str(np.round(np.mean(list_r2_dist_2_ce),4))+" ("+str(np.round(np.std(list_r2_dist_2_ce),4))+")"
text_rmse = str(np.round(np.mean(list_rmse_dist_2_ce),4))+" ("+str(np.round(np.std(list_rmse_dist_2_ce),4))+")"
text_crossentropy = str(np.round(np.mean(list_crossentropy_dist_2_ce),4))+" ("+str(np.round(np.std(list_crossentropy_dist_2_ce),4))+")"
text_similarity = str(np.round(np.mean(list_similarity_dist_2_ce),4))+" ("+str(np.round(np.std(list_similarity_dist_2_ce),4))+")"
text_rmse_a = str(np.round(np.mean(list_rmse_a_dist_2_ce),4))+" ("+str(np.round(np.std(list_rmse_a_dist_2_ce),4))+")"
table.add_row(["Distance", text_r2, text_rmse, text_crossentropy, text_similarity, text_rmse_a])

print(table.draw())

+-------------+------------+------------+------------+------------+------------+
| Model       | $R^2$      | RMSE       | Cross-     | Cos        | RMSE_A     |
|             |            |            | entropy    | similarity |            |
+-------------+------------+------------+------------+------------+------------+
| Not spatial | 0.6325     | 0.0869     | 0.8983     | 0.9008     | 0.6302     |
|             | (0.0151)   | (0.0018)   | (0.0031)   | (0.0062)   | (0.0149)   |
+-------------+------------+------------+------------+------------+------------+
| Contiguity  | 0.6302     | 0.0868     | 0.8971     | 0.9114     | 0.6167     |
|             | (0.0169)   | (0.002)    | (0.003)    | (0.0053)   | (0.0145)   |
+-------------+------------+------------+------------+------------+------------+
| Distance    | 0.6321     | 0.085      | 0.8958     | 0.9055     | 0.6117     |
|             | (0.017)    | (0.0024)   | (0.0028)   | (0.0076)   | (0.0171)   |
+-------------+------------+